# 02 · Relatório do Projeto
## Agente inteligente de rotas para distribuição de produtos locais

**Unidade Curricular:** Inteligência Artificial · **Projeto 01**
**Grupo:** G## · **Elementos:** Afonso Macedo (nº31554), Nome 2 (nº), Nome 3 (nº), Nome 4 (nº)
**Repositório GitHub:** *(colar aqui o link; é o mesmo que está no `readme.txt`)*

---

### Resumo
Este relatório descreve um agente que planeia as rotas de uma frota de veículos que **recolhe produtos em pequenos produtores, os leva ao armazém (centro de recolha) e depois os entrega aos clientes**, respeitando janelas temporais e a capacidade dos veículos, com o objetivo de **minimizar os quilómetros percorridos**.

Modelámos o problema, implementámos **Greedy, A\*, Hill Climbing e um CSP** (com MRV, forward checking e LCV) e comparámo-los no dataset *tiny* (19 pedidos planeáveis, 2 veículos). Principais resultados:
- todos os métodos produzem rotas **completas e viáveis**, exceto o A\* clássico, que não termina dentro do limite de 30 000 nós;
- a **melhor solução encontrada custa 1594,8 km** e resulta de construir uma solução com o **A\* ponderado (w = 1,5)** e refiná-la com **Hill Climbing**;
- a **pesquisa local** é a técnica que mais reduz o custo, e a **heurística** é decisiva para tornar a pesquisa informada praticável;
- a regra "**passar pelo armazém antes de entregar**" alonga as viagens (de cerca de 56 para cerca de 100 minutos por pedido) e torna **1 pedido impossível de cumprir** (o pedido 8).

### Índice
1. **Introdução** (contexto, problema, objetivos, glossário e como executar)
2. **Análise exploratória dos dados (EDA)**
3. **Modelo** (pressupostos, regra do armazém, restrições, custo, heurísticas e algoritmos)
4. **Implementação** (resultados de cada técnica e análise crítica)
5. **Conclusão**
6. **Anexos** (estrutura da entrega e referências)

---
## 1. Introdução

### 1.1 Contexto e motivação
A procura por alimentos locais está a crescer na Europa, mas os pequenos produtores enfrentam custos logísticos elevados e preços instáveis. Cadeias de abastecimento curtas reduzem as emissões de transporte e o desperdício alimentar, mas só são viáveis se a distribuição for **eficiente**.

Imagine-se uma plataforma digital que liga três tipos de atores:
- **produtores**, pequenos agricultores que vendem frutas, legumes e outros produtos;
- **clientes**, que fazem encomendas e escolhem um local de entrega;
- **centros de recolha** (armazéns), que agregam os produtos antes da distribuição e de onde parte e para onde regressa a frota de veículos.

### 1.2 O problema
Dado um conjunto de pedidos, cada um com um produtor de origem, um cliente de destino, um peso e **duas janelas temporais** (recolha e entrega), e uma frota de veículos com capacidade limitada, pretende-se **atribuir os pedidos aos veículos e ordenar as paragens** de forma a:
- servir **todos** os pedidos;
- respeitar as janelas temporais, a capacidade e a ordem **recolha → armazém → entrega**;
- **minimizar a distância total percorrida**.

Este é um problema clássico de investigação operacional, conhecido como *Pickup and Delivery Problem with Time Windows* (PDPTW), com a particularidade de os produtos terem de **passar pelo armazém** entre a recolha e a entrega. É computacionalmente difícil: o número de rotas possíveis cresce de forma explosiva com o número de pedidos.

### 1.3 Objetivos
Desenvolver um agente inteligente que resolva o problema com técnicas estudadas na unidade curricular, e compará-las:
1. **Pesquisa informada:** Greedy best-first e A\*, com duas heurísticas, contando os nós expandidos.
2. **Pesquisa local:** Hill Climbing, que melhora uma solução inicial explorando soluções vizinhas.
3. **Problema de Satisfação de Restrições (CSP):** atribuição de pedidos a veículos com backtracking e heurísticas.
4. **Análise crítica** dos resultados: custo, esforço de pesquisa e tempo de execução.

### 1.4 Organização do trabalho
| Ficheiro | Conteúdo |
|---|---|
| `01_eda.ipynb` | análise exploratória e validação dos dados |
| `algoritmos.py` | modelo do problema e todos os algoritmos (código reutilizável) |
| `02_relatorio.ipynb` | **este notebook**: relatório que explica, executa e analisa tudo |

### 1.5 Glossário
| Termo | Significado neste projeto |
|---|---|
| **Pedido** | um item de encomenda: recolher num produtor e entregar a um cliente |
| **Janela temporal** | intervalo em que a recolha (ou a entrega) pode acontecer |
| **Rota** | sequência de paragens de um veículo: `("P", item)` recolha, `("D", item)` entrega |
| **Solução** | as rotas de todos os veículos (`{veículo: rota}`) |
| **Viável** | uma rota que cumpre todas as restrições |
| **Custo** | quilómetros totais percorridos |
| **Heurística** | estimativa do custo que falta percorrer a partir de um estado |
| **Admissível** | heurística que nunca sobrestima o custo real |
| **Nó expandido** | estado cujos sucessores foram gerados pela pesquisa (mede o esforço) |
| **Ótimo local** | solução que nenhuma pequena alteração consegue melhorar |
| **CSP** | problema de satisfação de restrições: variáveis, domínios e restrições |
| **Retrocesso** | vez em que o backtracking desfaz uma atribuição por não ter saída |

### 1.6 Como executar este notebook
1. Ter na mesma pasta (ou carregar no Colab) o **`algoritmos.py`** e a pasta **`data_tiny/`** (ou o `data_tiny.zip`).
2. Executar todas as células por ordem (**Ambiente de execução → Executar tudo**). Demora cerca de 20 segundos.
3. Os resultados são **determinísticos** (não há aleatoriedade): custos e contagens de nós/retrocessos são sempre os mesmos; só os tempos variam de máquina para máquina.

#### Preparar o ambiente
Importamos as bibliotecas e o módulo `algoritmos.py`, e criamos o objeto `p` (da classe `Problem`), que carrega o dataset e é usado em todo o relatório. No Google Colab, se os ficheiros não existirem, a célula pede para os carregar.

In [ ]:
import os, zipfile, time, math, itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

DATA_DIR = "data_tiny/"          # mudar para "data_large/" para o dataset maior

# --- Google Colab: se faltar o algoritmos.py ou os dados, pede os ficheiros ---
if not (os.path.exists("algoritmos.py") and os.path.exists(DATA_DIR)):
    try:
        from google.colab import files
        print("Carregue o algoritmos.py e o data_tiny.zip")
        up = files.upload()
        for name in up:
            if name.endswith(".zip"):
                zipfile.ZipFile(name).extractall(".")
    except ImportError:
        raise FileNotFoundError("Faltam o algoritmos.py ou a pasta de dados")

from algoritmos import *          # Vehicle, Request, State, Problem, informed_search, ...

p = Problem(DATA_DIR)
print(f"✔ Problema carregado: {len(p.requests)} pedidos planeáveis e {len(p.vehicles)} veículos")

Carregue o algoritmos.py e o data_tiny.zip


---
## 2. Análise exploratória dos dados (EDA)

A análise exploratória completa está no notebook **`01_eda.ipynb`**, entregue em conjunto com este relatório. Lá estão as validações de integridade, as janelas temporais, os pesos e o mapa das paragens. Aqui resumimos apenas **o que descobrimos e como isso influenciou o modelo**.

| O que a EDA mostrou | Consequência para o modelo |
|---|---|
| Sem nulos, duplicados nem referências órfãs | os dados podem ser carregados e juntados sem limpeza |
| Matriz de distâncias completa, **simétrica** e com **desigualdade triangular** | as heurísticas baseadas em distâncias diretas são admissíveis |
| Os `location_id` da matriz **não coincidem** com `producer_id`, `customer_id` nem `center_id` | é preciso mapear `(tipo, ref_id)` → `location_id` através de `locations.csv` |
| 1 encomenda já `delivered` (3 itens) | fica **fora do planeamento**: restam 20 pedidos |
| Com a regra do armazém, **1 pedido (o 8) é impossível** mesmo sozinho | é retirado do planeamento: restam **19 pedidos** |
| Janelas de poucas horas, espalhadas por duas semanas (pico de 5 itens abertos ao mesmo tempo) | o veículo **espera** quando chega cedo; a espera **não tem custo**; os tempos são convertidos para minutos desde o primeiro instante |
| Peso total ~249 kg nos 23 itens (198 kg nos 19 pedidos planeáveis) contra capacidades de 1010 e 1445 kg | a capacidade **nunca é a restrição ativa** neste dataset, mas mantém-se no modelo |
| Matriz a 45 km/h; veículos a ~58–60 km/h | usamos o tempo da matriz (`duration_min`) como tempo de viagem |
| Nenhum produto indica necessidade de frio | o campo `refrigerated` não é usado |

As células seguintes confirmam estes números a partir do objeto `p` que os algoritmos vão usar.

In [ ]:
pesos = [r.weight for r in p.requests]
inicio = min(r.availability_window[0] for r in p.requests)
fim = max(r.delivery_window[1] for r in p.requests)
d = p.depot
direto = np.mean([p.travel(r.pickup_loc, r.delivery_loc)[1] for r in p.requests])
via = np.mean([p.travel(r.pickup_loc, d)[1] + p.travel(d, r.delivery_loc)[1] for r in p.requests])

print(f"Pedidos planeáveis:           {len(p.requests)}")
print(f"Pedidos retirados (impossíveis): {[r.item_id for r in p.dropped]}")
print(f"Veículos:                     {len(p.vehicles)}  (capacidades: {[v.capacity for v in p.vehicles]} kg)")
print(f"Centro de recolha (location): {p.depot}")
print(f"Peso total / maior pedido:    {sum(pesos):.1f} kg / {max(pesos):.1f} kg")
print(f"Horizonte temporal:           {p.t0} -> {p.t0 + pd.Timedelta(minutes=fim)}")
print(f"Duração do horizonte:         {(fim - inicio) / 1440:.1f} dias")
print(f"Viagem produtor->cliente (média):          {direto:.0f} min  (se fosse direta)")
print(f"Viagem produtor->armazém->cliente (média): {via:.0f} min  (regra do armazém)")

**O que vemos.**
- Os números coincidem com os da EDA. O peso total (198 kg) é inferior aos 249 kg da EDA porque já não inclui os 3 itens da encomenda entregue nem o pedido retirado.
- São 19 pedidos leves (cerca de 20% da capacidade do veículo mais pequeno, mesmo somados todos) espalhados por cerca de duas semanas. Como o peso não limita, o desafio está sobretudo em **cumprir as janelas temporais** e **ordenar bem as paragens**.
- Obrigar os produtos a **passar pelo armazém** alonga a viagem média de cada pedido de cerca de 56 para cerca de 100 minutos, o que aperta as janelas de entrega.

#### Pedido impossível
A tabela seguinte mostra o pedido que foi retirado do planeamento e porquê.

In [ ]:
linhas = []
for r in p.dropped:
    via_min = p.travel(r.pickup_loc, d)[1] + p.travel(d, r.delivery_loc)[1]
    chegada = r.availability_window[0] + via_min        # melhor caso: recolher logo à abertura
    linhas.append({"pedido": r.item_id, "encomenda": r.order_id,
                   "recolha abre (min)": round(r.availability_window[0]),
                   "viagem via armazém (min)": round(via_min),
                   "chegada mais cedo (min)": round(chegada),
                   "entrega fecha (min)": round(r.delivery_window[1]),
                   "atraso inevitável (min)": round(chegada - r.delivery_window[1])})
pd.DataFrame(linhas)

**O que vemos.** Mesmo recolhendo o produto no primeiro instante possível e viajando sem esperas, o veículo chega ao cliente **28 minutos depois** de a janela de entrega fechar. Nenhum algoritmo consegue resolver isto, por isso o pedido é retirado e **reportado**. (Entregando diretamente, sem passar pelo armazém, o pedido seria possível: a viagem direta dura 71 minutos em vez de 148.)

#### Distribuição dos pedidos no tempo
Cada linha é um pedido: a barra **azul** é a janela de recolha e a **laranja** a de entrega.

In [ ]:
ordenados = sorted(p.requests, key=lambda r: r.availability_window[0])
fig, ax = plt.subplots(figsize=(11, 5.5))
for k, r in enumerate(ordenados):
    a0, a1 = r.availability_window
    b0, b1 = r.delivery_window
    ax.hlines(k, a0 / 1440, a1 / 1440, color="tab:blue", lw=6, label="recolha" if k == 0 else None)
    ax.hlines(k, b0 / 1440, b1 / 1440, color="tab:orange", lw=6, label="entrega" if k == 0 else None)
ax.set_yticks(range(len(ordenados)))
ax.set_yticklabels([f"pedido {r.item_id}" for r in ordenados], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("dias desde o primeiro instante (t0)")
ax.set_title("Janelas de recolha (azul) e de entrega (laranja)")
ax.legend()
ax.grid(alpha=.3)
plt.tight_layout()
plt.show()

**O que vemos.** Os pedidos estão **espalhados ao longo de cerca de 14 dias** e as janelas de pedidos diferentes raramente se sobrepõem. Por isso o mesmo veículo pode servir muitos pedidos seguidos, **esperando** entre eles (a espera não tem custo no nosso modelo), e só há conflito real nos dias em que várias janelas coincidem.

#### Localização das paragens

In [ ]:
cores = {"producer": "tab:green", "collection_center": "tab:red", "delivery_location": "tab:blue"}
formas = {"producer": "o", "collection_center": "*", "delivery_location": "s"}
fig, ax = plt.subplots(figsize=(7.5, 6.5))
for t, g in p.locations.groupby("location_type"):
    ax.scatter(g.longitude, g.latitude, c=cores[t], marker=formas[t],
               s=300 if t == "collection_center" else 90, label=t, edgecolor="k", zorder=3)
    for _, r in g.iterrows():
        ax.annotate(str(r.location_id), (r.longitude, r.latitude), xytext=(5, 5),
                    textcoords="offset points", fontsize=9)
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.grid(alpha=.3)
ax.legend()
ax.set_title("Paragens (número = location_id)")
plt.tight_layout()
plt.show()

**O que vemos.** São 12 paragens: 1 centro, 5 produtores e 6 locais de entrega, espalhados pelo distrito de Braga sem uma estrutura em estrela ou em linha. Os produtores 1 e 5 têm as mesmas coordenadas. As distâncias ao centro vão de cerca de 9 a 65 km, por isso o custo total das rotas é dominado por **para onde** se vai e não por quantas paragens se fazem.

---
## 3. Modelo

Nesta secção explicamos **como o problema foi modelado**. Todo o código está no ficheiro **`algoritmos.py`**; a tabela seguinte indica onde está cada conceito.

| Conceito do modelo | Onde está em `algoritmos.py` |
|---|---|
| Veículo (capacidade, velocidade, centro) | classe `Vehicle` |
| Pedido (recolha + entrega, peso, 2 janelas) | classe `Request` |
| Estado da pesquisa | classe `State` |
| Carregamento dos dados e regras do problema | classe `Problem` |
| Uma paragem (aplica todas as regras num só sítio) | `Problem.move` |
| Verificação de viabilidade de uma rota | `Problem.route_feasibility_checker` |
| Função de custo | `Problem.cost_function` |
| Heurísticas | `Problem.h_admissible` e `Problem.h_informative` |
| Greedy e A\* | `informed_search` (com `_successors` e `_dead`) |
| Hill Climbing | `hill_climbing` |
| CSP (backtracking, MRV, forward checking, LCV) | `solve_csp` e `find_sequence` |

### 3.1 Pressupostos e decisões de modelação
| # | Decisão | Justificação |
|---|---|---|
| 1 | O veículo **parte e regressa ao centro de recolha** | é o modelo de negócio (armazém com frota) |
| 2 | **Regra do armazém:** o produto tem de passar pelo armazém entre a recolha e a entrega | o armazém agrega os produtos antes da distribuição |
| 3 | O mesmo veículo que recolhe um item é o que o entrega | simplifica o modelo; uma extensão seria permitir transferências entre veículos |
| 4 | Os tempos são **minutos desde o primeiro instante** `t0` | comparar janelas passa a ser comparar números |
| 5 | **Chegar cedo é esperar**, sem custo; chegar tarde é inviável | as janelas são rígidas |
| 6 | O custo é a **distância percorrida** (km) | é o objetivo do enunciado |
| 7 | Tempo de viagem = `duration_min` da matriz | é a fonte de tempos do dataset |
| 8 | Sem tempo de serviço (carga/descarga) | o dataset não o indica |
| 9 | Encomendas `delivered` ficam fora do planeamento | já foram entregues |
| 10 | Pedidos impossíveis sozinhos são retirados e reportados | nenhum algoritmo os consegue servir |
| 11 | Os veículos são construídos **um de cada vez** | evita explorar a mesma solução em muitas ordens (simetrias) |

### 3.2 Representação
- **Pedido:** cada item de encomenda é um `Request` com o local de recolha (produtor), o local de entrega (cliente), o peso (`quantidade × peso unitário`) e as duas janelas temporais, em minutos desde `t0`.
- **Rota:** uma lista de paragens `[("P", item), ("D", item), ...]`. A passagem pelo armazém **não aparece escrita** na rota: é implícita (ver 3.3).
- **Solução:** um dicionário `{veículo: rota}`. Está **completa** quando todos os pedidos são entregues exatamente uma vez.

### 3.3 A regra do armazém
O veículo **não pode entregar diretamente** o que recolheu: tem de passar pelo armazém antes. No modelo, isto é aplicado assim:
- ao ir **entregar um item que ainda não passou pelo armazém**, a viagem faz-se obrigatoriamente `local atual → armazém → cliente`;
- ao passar pelo armazém, **tudo o que vai a bordo fica "armazenado"** e pode ser entregue diretamente nas paragens seguintes, sem nova passagem.

Isto permite agrupar: por exemplo, recolher dois itens e entregar ambos com **uma só** ida ao armazém. A função `detalhar_rota` (célula seguinte) mostra uma rota perna a perna, com as passagens pelo armazém que o modelo acrescenta sozinho.

In [ ]:
def detalhar_rota(p, vid, route):
    """Mostra uma rota perna a perna, incluindo as passagens implícitas pelo armazém."""
    v = p.veh_by_id[vid]
    loc, onboard, armazenado, linhas = v.depot_loc, set(), set(), []

    def perna(de, para, nota):
        dk, dm = p.travel(de, para)
        linhas.append({"de": de, "para": para, "km": dk, "min": round(dm), "nota": nota})

    for kind, i in route:
        r = p.req_by_id[i]
        if kind == "P":
            perna(loc, r.pickup_loc, f"recolher pedido {i}")
            onboard.add(i)
            loc = r.pickup_loc
        else:
            if i not in armazenado:
                perna(loc, v.depot_loc, "passagem obrigatória pelo armazém")
                loc = v.depot_loc
                armazenado |= onboard
            perna(loc, r.delivery_loc, f"entregar pedido {i}")
            onboard.discard(i)
            armazenado.discard(i)
            loc = r.delivery_loc
    perna(loc, v.depot_loc, "regresso ao centro")
    return pd.DataFrame(linhas)


# procura dois pedidos que se possam recolher seguidos e entregar com uma só passagem pelo armazém
v1 = p.vehicles[0]
for a, b in itertools.combinations(sorted(p.requests, key=lambda r: r.availability_window[0]), 2):
    demo = [("P", a.item_id), ("P", b.item_id), ("D", a.item_id), ("D", b.item_id)]
    if p.route_feasibility_checker(demo, v1):
        break
print("Rota de demonstração:", demo)
pernas = detalhar_rota(p, v1.vehicle_id, demo)
display(pernas.round(1))
print(f"Total das pernas: {pernas.km.sum():.1f} km | cost_function: {p.cost_function(demo, v1):.1f} km")

**O que vemos.** Os dois pedidos são recolhidos seguidos e a rota faz **uma única passagem pelo armazém**, a seguir à segunda recolha: o primeiro pedido a entregar obriga a ir ao armazém e, a partir daí, ambos estão "armazenados", pelo que o segundo é entregue sem nova passagem. A soma das pernas coincide com o valor da `cost_function`, o que confirma que a função de custo aplica a regra corretamente.

### 3.4 Restrições e verificador de viabilidade
Uma rota só é **viável** se cumprir todas as restrições, que o `route_feasibility_checker` verifica simulando a rota passo a passo (através de `Problem.move`):

| # | Restrição | Como é tratada |
|---|---|---|
| R1 | **Janela de recolha** | se o veículo chega antes de abrir, espera; se chega depois de fechar, a rota é inviável |
| R2 | **Janela de entrega** | igual à anterior, já contando a passagem pelo armazém |
| R3 | **Capacidade** | a carga a bordo nunca pode exceder a capacidade, em **nenhum instante** (não só no total) |
| R4 | **Precedência** | só se entrega o que já foi recolhido |
| R5 | **Armazém** | antes de entregar, o item tem de passar pelo armazém |
| R6 | **Completude** | no fim da rota não pode ficar nada por entregar; o veículo regressa ao centro |

A `cost_function` devolve os **quilómetros** da rota (incluindo as passagens pelo armazém e o regresso ao centro) ou **infinito** se for inviável, para que nenhum algoritmo prefira uma rota impossível. A célula seguinte demonstra o verificador com quatro rotas: uma válida e três com erros diferentes.

In [ ]:
pedidos = sorted(p.requests, key=lambda r: r.availability_window[0])
cedo, tarde = pedidos[0], pedidos[-1]

exemplos = {
    "Pedido mais cedo, depois o mais tarde (válida)":
        [("P", cedo.item_id), ("D", cedo.item_id), ("P", tarde.item_id), ("D", tarde.item_id)],
    "Entregar antes de recolher (R4)":
        [("D", cedo.item_id), ("P", cedo.item_id)],
    "Ir primeiro ao mais tarde e depois ao mais cedo (R1: janela já fechou)":
        [("P", tarde.item_id), ("D", tarde.item_id), ("P", cedo.item_id), ("D", cedo.item_id)],
    "Recolher e nunca entregar (R6)":
        [("P", cedo.item_id)],
}
pd.DataFrame([{"rota": nome, "viável": p.route_feasibility_checker(rota, v1),
               "custo (km)": round(p.cost_function(rota, v1), 1)}
              for nome, rota in exemplos.items()])

**O que vemos.** A primeira rota é viável (ida e volta ao centro passando pelos dois pedidos, que estão em extremos opostos do horizonte, e **cada um com a sua passagem pelo armazém**). As outras três são rejeitadas, com custo infinito, cada uma por violar uma restrição diferente. É este verificador que garante que **todas as soluções produzidas pelos algoritmos são válidas**.

### 3.5 Função de custo
O custo de uma rota é a **soma dos quilómetros** de todas as pernas: o veículo parte do centro, percorre as paragens (com as passagens pelo armazém quando necessárias) e regressa ao centro. O custo de uma solução é a soma do custo das rotas dos veículos. A célula anterior (`detalhar_rota`) mostra esta soma perna a perna.

### 3.6 Espaço de estados e como o reduzimos
Com 19 pedidos há 38 paragens por ordenar (uma recolha e uma entrega por pedido). A célula seguinte calcula quantas ordens existem **só para um veículo**, contando apenas a regra de a recolha vir antes da entrega.

In [ ]:
n = len(p.requests)
ordens = math.factorial(2 * n) / 2 ** n      # 2n paragens; em cada par, recolha antes da entrega
print(f"{n} pedidos -> {2*n} paragens -> cerca de {ordens:.2e} ordens possíveis (ainda sem repartir pelos veículos)")

**O que vemos.** O número de ordens possíveis é astronómico (da ordem de 10³⁹), por isso é impossível testá-las todas. Três ideias tornam a pesquisa praticável:
1. **Construir um veículo de cada vez**, o que elimina as simetrias entre veículos.
2. **Poda de estados sem saída** (`_dead`): um estado é descartado se algum item a bordo já não chega a tempo à entrega (contando a passagem pelo armazém) ou, no último veículo, se algum pedido já não puder ser recolhido.
3. **Poda de estados repetidos**: se já foi expandido um estado equivalente (mesmo veículo, local, itens recolhidos, entregues e armazenados) com custo e hora iguais ou melhores, o novo é descartado.

### 3.7 Formulação como problema de pesquisa
| Elemento | Definição |
|---|---|
| **Estado** | veículo atual, local, hora, carga, itens recolhidos, entregues e **armazenados** (`State`) |
| **Estado inicial** | primeiro veículo parado no centro, às 0 h, vazio, sem nada recolhido |
| **Ações** | (a) recolher um item, (b) entregar um item que vai a bordo (com passagem pelo armazém se for preciso), (c) **fechar a rota** (regressar ao centro e passar ao veículo seguinte) |
| **Teste de objetivo** | todos os itens entregues e a última rota fechada |
| **Custo de cada ação** | quilómetros percorridos |

### 3.8 Heurísticas para a pesquisa informada
Uma heurística estima o custo que falta percorrer a partir de um estado. Usamos duas, ambas com a regra do armazém:

**Heurística 1, `h_admissible` (admissível).** Para cada pedido por entregar calcula-se o mínimo que *alguém* tem de andar para o servir e toma-se o **máximo** desses valores:
- por recolher: local → produtor → armazém → cliente → centro;
- recolhido mas ainda não armazenado: local → armazém → cliente → centro;
- já armazenado: local → cliente → centro.

Usa-se o máximo e não a soma porque uma mesma viagem pode servir vários pedidos: a soma sobrestimaria o custo real. Como a matriz cumpre a desigualdade triangular (ver EDA), esta heurística **nunca sobrestima** e por isso o A\* clássico com ela garantiria a solução ótima.

**Heurística 2, `h_informative` (não admissível).** Soma, para cada pedido por servir, a viagem que lhe falta (produtor → armazém → cliente, ou armazém → cliente se já vai a bordo) e junta a distância à paragem mais próxima. **Pode sobrestimar** (não tem em conta que uma viagem pode servir vários pedidos), por isso o A\* deixa de garantir otimalidade, mas orienta muito melhor a pesquisa.

Os dois algoritmos usam estas heurísticas de forma diferente:
- **Greedy best-first:** ordena os nós só por `f = h` (ignora o que já se gastou);
- **A\*:** ordena por `f = g + w·h`, onde `g` é o custo já percorrido. Com `w = 1` é o A\* clássico; com `w > 1` é o **A\* ponderado**, que dá mais peso à heurística, é mais rápido, mas já não garante a solução ótima.

A célula seguinte compara o valor das duas heurísticas no estado inicial.

In [ ]:
vazio = frozenset()
h1 = p.h_admissible(0, p.depot, vazio, vazio, vazio)
h2 = p.h_informative(0, p.depot, vazio, vazio, vazio)
print(f"Estimativa da heurística admissível no estado inicial:  {h1:8.1f} km")
print(f"Estimativa da heurística informativa no estado inicial: {h2:8.1f} km")

**O que vemos.** As duas heurísticas dão estimativas muito diferentes no estado inicial: a admissível estima apenas cerca de **223 km** (só vê o pior pedido isolado), enquanto a informativa estima cerca de **1448 km**. Como a melhor solução que vamos encontrar custa cerca de 1595 km, a admissível fica a uma enorme distância do custo real (cerca de 14%) e a informativa bem mais perto (cerca de 91%). Esta diferença ajuda a explicar, na secção 4, porque é que as duas se comportam de maneira tão distinta.

### 3.9 Os algoritmos em pseudo-código

**Greedy best-first e A\*** (`informed_search`)
```
fronteira ← {estado inicial}
enquanto a fronteira não está vazia:
    escolhe o nó n com menor f        (Greedy: f = h ; A*: f = g + w·h)
    se n é objetivo:  devolve as rotas
    se n é equivalente a um estado já expandido com custo e hora ≤:  ignora
    expande n: para cada ação válida (recolher, entregar, fechar rota):
        gera o filho; descarta-o se for um estado sem saída
        junta-o à fronteira
se esgotou o limite de nós: devolve "limite atingido"
```

**Hill Climbing** (`hill_climbing`)
```
solução ← solução inicial (do Greedy)
repetir:
    para cada pedido e para cada posição possível noutra rota (ou na mesma):
        calcula o ganho de mudar o pedido para essa posição (só se for viável)
    se nenhum movimento reduz o custo:  pára (ótimo local)
    aplica o movimento com maior ganho
```

**CSP com backtracking** (`solve_csp`)
```
atribuir(atribuições):
    se todos os pedidos têm veículo:  devolve a solução
    escolhe o pedido com menos veículos possíveis        (MRV)
    para cada veículo, do que menos km acrescenta ao que mais:   (LCV)
        se o conjunto de pedidos desse veículo não tem sequência viável:  retrocede
        atribui; remove dos outros pedidos os veículos que ficaram incompatíveis (forward checking)
        se nenhum domínio ficou vazio:  atribuir(...)
        desfaz a atribuição (retrocesso)
```

### 3.10 Pesquisa local: Hill Climbing
O Hill Climbing parte de uma solução completa e tenta melhorá-la com pequenas alterações.
- **Vizinhança "relocate":** retira um pedido (a recolha e a entrega) da rota onde está e reinsere-o noutra posição, no mesmo veículo ou no outro. Só se consideram soluções **viáveis**.
- **Critério:** em cada iteração examina todos os vizinhos e aplica o movimento que mais reduz o custo (*best improvement*).
- **Paragem:** quando nenhum vizinho melhora, a solução está num **ótimo local**.

### 3.11 Problema de Satisfação de Restrições (CSP)
| Elemento | Definição |
|---|---|
| **Variáveis** | uma por pedido: `X_i` = veículo que serve o pedido `i` |
| **Domínios** | `D(X_i)` = veículos que conseguem servir o pedido `i` pelo menos sozinhos |
| **Restrições duras** | para cada veículo, o conjunto de pedidos que lhe foi atribuído tem de admitir **uma sequência de paragens viável** (capacidade, janelas, precedência e armazém). É verificado por `find_sequence`, uma pesquisa em profundidade que tenta primeiro a paragem cujo prazo fecha mais cedo |
| **Restrição suave** | minimizar os quilómetros (não invalida soluções, só as ordena) |

**Heurísticas usadas** (cada uma pode ser ligada ou desligada para comparar):
- **MRV** (*Minimum Remaining Values*): escolhe a variável com menos valores possíveis, ou seja, a mais difícil primeiro; em empate, a de janela de entrega mais apertada.
- **Forward checking:** depois de cada atribuição remove, dos domínios dos pedidos ainda por atribuir, os veículos que deixaram de ser compatíveis; se algum domínio fica vazio, retrocede logo.
- **LCV** (*Least Constraining Value*): tenta primeiro o veículo que **aumenta menos os quilómetros**. É assim que tratamos a **restrição suave**.

---
## 4. Implementação

Nesta secção **executamos** os algoritmos de `algoritmos.py` sobre o dataset *tiny* e analisamos os resultados.

### 4.1 Funções auxiliares
Definimos funções só para **apresentar e verificar** resultados (não fazem parte do modelo):
- `mostrar_rotas`: horário de cada rota (chegada, hora de serviço, espera e carga em cada paragem, incluindo as passagens pelo armazém);
- `desenhar_rotas`: desenha as rotas no mapa;
- `resumo_rotas`: km e número de pedidos por veículo;
- `auditar`: **verificação independente** de uma solução. Refaz a simulação do zero, sem usar `Problem.move`, e confirma janelas, capacidade, regra do armazém e completude, para termos a certeza de que os algoritmos não "enganam" o verificador.

In [ ]:
def sequencia_locais(p, vid, route):
    """Locais visitados por um veículo, incluindo as passagens implícitas pelo armazém."""
    v = p.veh_by_id[vid]
    seq, onboard, arm = [v.depot_loc], set(), set()
    for kind, i in route:
        r = p.req_by_id[i]
        if kind == "P":
            seq.append(r.pickup_loc)
            onboard.add(i)
        else:
            if i not in arm:
                seq.append(v.depot_loc)
                arm |= onboard
            seq.append(r.delivery_loc)
            onboard.discard(i)
            arm.discard(i)
    seq.append(v.depot_loc)
    return seq


def mostrar_rotas(p, routes):
    """Tabela com o horário de cada veículo, paragem a paragem."""
    rows = []
    for vid, route in routes.items():
        v = p.veh_by_id[vid]
        loc, t, carga, onboard, arm = v.depot_loc, 0.0, 0.0, set(), set()

        def linha(rotulo, local, chegada, servico):
            rows.append({"veículo": vid, "paragem": rotulo, "local": local,
                         "chegada": (p.t0 + pd.Timedelta(minutes=chegada)).round("min"),
                         "serviço": (p.t0 + pd.Timedelta(minutes=servico)).round("min"),
                         "espera (min)": round(servico - chegada),
                         "carga (kg)": round(max(carga, 0.0), 1)})

        for kind, i in route:
            r = p.req_by_id[i]
            if kind == "P":
                t += p.travel(loc, r.pickup_loc)[1]
                cheg = t
                t = max(t, r.availability_window[0])
                carga += r.weight
                onboard.add(i)
                linha(f"P{i}", r.pickup_loc, cheg, t)
                loc = r.pickup_loc
            else:
                if i not in arm:
                    t += p.travel(loc, v.depot_loc)[1]
                    loc = v.depot_loc
                    arm |= onboard
                    linha("armazém", loc, t, t)
                t += p.travel(loc, r.delivery_loc)[1]
                cheg = t
                t = max(t, r.delivery_window[0])
                carga -= r.weight
                onboard.discard(i)
                arm.discard(i)
                linha(f"D{i}", r.delivery_loc, cheg, t)
                loc = r.delivery_loc
    return pd.DataFrame(rows)


def desenhar_rotas(p, routes, titulo):
    """Desenha no mapa a sequência de locais visitada por cada veículo."""
    L = p.locations.set_index("location_id")
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(L.longitude, L.latitude, c="lightgray", edgecolor="k", zorder=3)
    for lid, row in L.iterrows():
        ax.annotate(str(lid), (row.longitude, row.latitude), xytext=(4, 4),
                    textcoords="offset points", fontsize=8)
    cores = ["tab:red", "tab:blue", "tab:green"]
    for k, (vid, route) in enumerate(routes.items()):
        if not route:
            continue
        seq = sequencia_locais(p, vid, route)
        ax.plot([L.loc[s, "longitude"] for s in seq], [L.loc[s, "latitude"] for s in seq],
                color=cores[k % 3], alpha=.5, lw=1.5, label=f"veículo {vid}", zorder=2)
    ax.scatter(L.loc[p.depot, "longitude"], L.loc[p.depot, "latitude"], marker="*",
               s=300, c="gold", edgecolor="k", zorder=4, label="armazém")
    ax.set_title(titulo)
    ax.legend()
    ax.grid(alpha=.3)
    plt.tight_layout()
    plt.show()


def resumo_rotas(p, routes):
    """km e nº de pedidos por veículo."""
    return pd.DataFrame([{"veículo": vid,
                          "pedidos": sum(1 for k, _ in rt if k == "P"),
                          "km": round(p.cost_function(rt, p.veh_by_id[vid]), 1)}
                         for vid, rt in routes.items()])


def auditar(p, routes):
    """Verificação independente (não usa Problem.move). Devolve (tabela, lista de erros)."""
    erros, linhas, entregues, dep = [], [], set(), p.depot
    for vid, rt in routes.items():
        v = p.veh_by_id[vid]
        loc, t, km, carga, pico, espera, idas = v.depot_loc, 0.0, 0.0, 0.0, 0.0, 0.0, 0
        onb, arm = set(), set()
        for k, i in rt:
            r = p.req_by_id[i]
            if k == "P":
                dk, dm = p.travel(loc, r.pickup_loc)
                km += dk
                t += dm
                a0, a1 = r.availability_window
                if t < a0:
                    espera += a0 - t
                    t = a0
                if t > a1 + 1e-9:
                    erros.append(f"recolha do pedido {i} fora da janela")
                carga += r.weight
                onb.add(i)
                loc = r.pickup_loc
                pico = max(pico, carga)
                if carga > v.capacity:
                    erros.append(f"capacidade excedida no pedido {i}")
            else:
                if i not in onb:
                    erros.append(f"pedido {i} entregue sem recolha")
                if i not in arm:                       # tem de passar pelo armazém
                    dk, dm = p.travel(loc, dep)
                    km += dk
                    t += dm
                    loc = dep
                    arm |= onb
                    idas += 1
                dk, dm = p.travel(loc, r.delivery_loc)
                km += dk
                t += dm
                b0, b1 = r.delivery_window
                if t < b0:
                    espera += b0 - t
                    t = b0
                if t > b1 + 1e-9:
                    erros.append(f"entrega do pedido {i} fora da janela")
                carga -= r.weight
                onb.discard(i)
                arm.discard(i)
                entregues.add(i)
                loc = r.delivery_loc
        if rt:
            dk, dm = p.travel(loc, dep)
            km += dk
            t += dm
        if onb:
            erros.append(f"veículo {vid}: pedidos {sorted(onb)} nunca entregues")
        if rt and abs(km - p.cost_function(rt, v)) > 1e-6:
            erros.append(f"veículo {vid}: custo diverge do verificador")
        linhas.append({"veículo": vid, "pedidos": len(rt) // 2, "km": round(km, 1),
                       "pico de carga (kg)": round(pico, 1), "capacidade (kg)": v.capacity,
                       "idas forçadas ao armazém": idas,
                       "espera total (dias)": round(espera / 1440, 1),
                       "regressa ao centro (dia)": round(t / 1440, 1)})
    if entregues != set(p.req_by_id):
        erros.append(f"pedidos por entregar: {sorted(set(p.req_by_id) - entregues)}")
    return pd.DataFrame(linhas), erros

### 4.2 Pesquisa informada: Greedy e A\*
Executamos o Greedy e o A\* com as duas heurísticas. Para o A\* usamos ainda a versão **ponderada** (`w = 1,5` e `w = 2`). Cada execução tem um limite de **30 000 nós expandidos** para não ficar presa em instâncias difíceis. Para cada uma registamos o **estado** (resolvido ou limite atingido), o **custo** da solução, o **número de nós expandidos** (a métrica pedida no enunciado) e o **tempo**.

In [ ]:
configs = [("greedy", "admissible", 1), ("greedy", "informative", 1),
           ("astar",  "admissible", 1), ("astar",  "informative", 1),
           ("astar",  "informative", 1.5), ("astar",  "informative", 2)]
H = {"admissible": "h admissível", "informative": "h informativa"}

resultados = {}
linhas = []
for algo, h, w in configs:
    r = informed_search(p, algo, h, w=w, max_expansions=30_000)
    nome = ("Greedy" if algo == "greedy" else ("A*" if w == 1 else f"A* ponderado (w={w})")) + f" | {H[h]}"
    resultados[nome] = r
    linhas.append({"algoritmo": nome, "estado": r.status,
                   "custo (km)": None if r.status != "solved" else round(r.cost, 1),
                   "nós expandidos": r.expanded, "tempo (s)": round(r.seconds, 2)})
pd.DataFrame(linhas)

**O que vemos.**
- O **Greedy** resolve sempre e muito depressa, mas as soluções são de qualidade fraca: 1983 km com a heurística admissível e 1866 km com a informativa. Com a informativa expande apenas **40 nós** contra **892** da admissível, ou seja, uma heurística mais informada reduz drasticamente o esforço de pesquisa **e** melhora a solução.
- O **A\* clássico (`w = 1`) não termina** com nenhuma das duas heurísticas: esgota o limite de 30 000 nós sem encontrar solução. Uma das causas é que o custo real das rotas (mais de 1500 km) é muito maior do que o que as heurísticas estimam no início (223 km e 1448 km), por isso o A\* tem poucas pistas e explora enormes regiões do espaço de estados.
- O **A\* ponderado com a heurística informativa** resolve com muito poucos nós e dá as melhores soluções construtivas: **1661,5 km com `w = 1,5` (56 nós)** e **1729,8 km com `w = 2` (44 nós)**, bastante melhores do que os dois Greedy.

Como o A\* clássico não termina, **não conseguimos provar qual é a solução ótima**.

#### Efeito do peso `w` no A\*
O parâmetro `w` controla o equilíbrio entre qualidade e rapidez. Variamo-lo nas duas heurísticas (os gráficos mostram a informativa, a única que resolve).

In [ ]:
varrer = []
for h in ["informative", "admissible"]:
    for w in [1, 1.5, 2, 3, 5, 10]:
        r = informed_search(p, "astar", h, w=w, max_expansions=30_000)
        varrer.append({"heurística": H[h], "w": w, "estado": r.status,
                       "custo (km)": None if r.status != "solved" else round(r.cost, 1),
                       "nós expandidos": r.expanded})
tab_w = pd.DataFrame(varrer)
display(tab_w)

ok = tab_w[(tab_w.estado == "solved") & (tab_w["heurística"] == "h informativa")]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(ok["w"], ok["custo (km)"], marker="o")
ax[0].set_title("Custo da solução vs. w")
ax[0].set_xlabel("w")
ax[0].set_ylabel("km")
ax[1].plot(ok["w"], ok["nós expandidos"], marker="o", color="tab:orange")
ax[1].set_title("Nós expandidos vs. w")
ax[1].set_xlabel("w")
for a in ax:
    a.grid(alpha=.3)
plt.tight_layout()
plt.show()

**O que vemos.** Há um ponto ótimo: com `w = 1` a pesquisa não termina, com `w = 1,5` já resolve com a **melhor solução** (1661,5 km) e, a partir daí, aumentar `w` torna a pesquisa ligeiramente mais rápida mas **piora a solução** (1729,8 km com `w = 2`, 1800,7 km com `w = 3` e 1866,5 km com `w = 10`, que é o valor do Greedy). Quando `w` é grande o A\* comporta-se como o Greedy, porque o custo já percorrido deixa de pesar. Por isso o `w` é um **compromisso** entre qualidade e esforço. Com a heurística **admissível**, nenhum valor de `w` chega a resolver dentro do limite: a estimativa é tão baixa que, mesmo multiplicada por 10, continua a orientar mal a pesquisa.

#### Rotas obtidas com o A\* ponderado (`w = 1,5`)

In [ ]:
sol_astar = resultados["A* ponderado (w=1.5) | h informativa"]
print(f"Custo total: {sol_astar.cost:.1f} km | solução completa: {p.is_complete(sol_astar.routes)}")
display(resumo_rotas(p, sol_astar.routes))
desenhar_rotas(p, sol_astar.routes, "A* ponderado (w=1.5), heurística informativa")
mostrar_rotas(p, sol_astar.routes).head(14)

**O que vemos.** A solução é **completa e viável**: os 19 pedidos são entregues, repartidos pelos dois veículos. O mapa mostra muitas idas e voltas ao armazém (a estrela), consequência da regra do armazém. A tabela mostra o horário (só as primeiras paragens): as linhas "armazém" são as passagens obrigatórias, o veículo **espera** quando chega antes de a janela abrir e a carga a bordo é sempre muito inferior à capacidade, confirmando o que a EDA tinha indicado.

### 4.3 Pesquisa local: Hill Climbing
Aplicamos o Hill Climbing sobre a solução do **Greedy**, como pedido no enunciado. Para ver o efeito do ponto de partida, partimos também da melhor solução do A\* ponderado. Registamos o custo depois de cada iteração e desenhamos a curva **custo vs. iteração**.

In [ ]:
inicios = {"Greedy | h informativa":              resultados["Greedy | h informativa"],
           "Greedy | h admissível":               resultados["Greedy | h admissível"],
           "A* ponderado (w=1.5) | h informativa": resultados["A* ponderado (w=1.5) | h informativa"]}

hc_rotas, hc_hist, hc_tempo, linhas = {}, {}, {}, []
for nome, base in inicios.items():
    t0 = time.time()
    rotas, hist = hill_climbing(p, base.routes)
    hc_rotas[nome], hc_hist[nome] = rotas, hist
    hc_tempo[nome] = time.time() - t0
    linhas.append({"ponto de partida": nome, "custo inicial (km)": round(hist[0], 1),
                   "custo final (km)": round(hist[-1], 1),
                   "melhoria": f"{(hist[0] - hist[-1]) / hist[0]:.1%}",
                   "iterações": len(hist) - 1, "tempo (s)": round(hc_tempo[nome], 1)})
display(pd.DataFrame(linhas))

plt.figure(figsize=(7.5, 4))
for nome, hist in hc_hist.items():
    plt.plot(range(len(hist)), hist, marker="o", label=nome)
plt.xlabel("iteração")
plt.ylabel("custo total (km)")
plt.title("Hill Climbing: custo vs. iteração")
plt.grid(alpha=.3)
plt.legend()
plt.tight_layout()
plt.show()

**O que vemos.**
- O Hill Climbing **melhora sempre** a solução inicial: cerca de **19%** a partir do Greedy admissível (1983 → 1601 km), **12%** a partir do Greedy informativo (1866 → 1642 km) e **4%** a partir do A\* ponderado (1661 → 1595 km). A melhoria é **monótona**, porque só aceitamos movimentos que reduzem o custo, e as primeiras iterações são as que mais poupam.
- Termina em poucas iterações (4 a 8) e em cerca de um segundo, quando atinge um **ótimo local**: nenhuma realocação de um só pedido melhora mais.
- Quanto **pior** é a solução inicial, **maior** é a melhoria (o Greedy admissível melhora 19%), mas os três pontos de partida convergem para custos parecidos (1595 a 1642 km). Partir de uma boa solução (A\* ponderado) dá o **melhor resultado final: 1594,8 km**.

Como o Hill Climbing só consegue descer, não sabemos o quão perto está do ótimo global.

### 4.4 CSP: atribuição de pedidos a veículos
Resolvemos o CSP descrito na secção 3.11 com **quatro configurações** de heurísticas, de modo a medir o contributo de cada uma. Medimos o **número de retrocessos** (a métrica pedida no enunciado), o número de atribuições tentadas, o custo da solução e o tempo.

In [ ]:
configs_csp = [("Backtracking simples",               dict(use_mrv=False, use_forward_checking=False, use_lcv=False)),
               ("+ MRV",                              dict(use_mrv=True,  use_forward_checking=False, use_lcv=False)),
               ("+ MRV + forward checking",           dict(use_mrv=True,  use_forward_checking=True,  use_lcv=False)),
               ("+ MRV + forward checking + LCV",     dict(use_mrv=True,  use_forward_checking=True,  use_lcv=True))]

csp_res, linhas = {}, []
for nome, cfg in configs_csp:
    c = solve_csp(p, **cfg)
    csp_res[nome] = c
    linhas.append({"configuração": nome, "estado": c.status, "custo (km)": round(c.cost, 1),
                   "atribuições tentadas": c.nodes, "retrocessos": c.backtracks,
                   "tempo (s)": round(c.seconds, 2)})
pd.DataFrame(linhas)

**O que vemos.**
- **Todas as configurações encontram uma solução viável** logo à primeira, sem **nenhum retrocesso**, e **todas dão o mesmo resultado** (1774,8 km). Isto é um resultado honesto, mas que importa explicar: neste dataset as janelas dos pedidos estão muito espalhadas no tempo, por isso **um só veículo consegue servir todos os pedidos** e nunca há conflitos que obriguem a retroceder. Sem conflitos, o MRV, o forward checking e o LCV não têm onde mostrar vantagem.
- Isto **não significa que as heurísticas sejam inúteis**: com um dataset mais apertado (mais pedidos concorrentes, janelas mais estreitas) é nesses casos que o forward checking evita retrocessos. Só o dataset maior permitiria demonstrá-lo.
- Como o custo é só a distância e nenhuma restrição obriga a equilibrar a carga, a **restrição suave** (minimizar km) empurra tudo para o mesmo veículo.
- O CSP resolve a **atribuição** de pedidos a veículos, mas a sequência de paragens de cada veículo é simplesmente a primeira viável que a pesquisa encontra, por isso o custo (1774,8 km) é pior do que o do A\* ponderado (1661,5 km). É natural aplicar depois o Hill Climbing para melhorar as sequências.

In [ ]:
csp = csp_res["+ MRV + forward checking + LCV"]
atrib = pd.DataFrame({"pedido": list(csp.assignment.keys()),
                      "veículo": list(csp.assignment.values())}).sort_values("pedido")
print("Atribuição de pedidos a veículos (resultado do CSP):")
display(atrib.groupby("veículo")["pedido"].apply(list).to_frame("pedidos"))

# melhorar as sequências do CSP com Hill Climbing
t0 = time.time()
csp_hc_rotas, csp_hc_hist = hill_climbing(p, csp.routes)
csp_hc_tempo = time.time() - t0
print(f"CSP: {csp.cost:.1f} km  ->  CSP + Hill Climbing: {csp_hc_hist[-1]:.1f} km "
      f"({len(csp_hc_hist) - 1} iterações, {csp_hc_tempo:.1f} s)")
display(resumo_rotas(p, csp_hc_rotas))
desenhar_rotas(p, csp_hc_rotas, "CSP + Hill Climbing")

**O que vemos.** O CSP coloca os 19 pedidos no veículo 1 (o veículo 2 fica parado). Depois de aplicar o Hill Climbing, o custo desce de **1774,8 km para 1606,1 km** (cerca de 9,5%) e os pedidos passam a repartir-se pelos dois veículos (15 e 4), mostrando que os dois métodos se complementam: o CSP garante uma atribuição viável e o Hill Climbing refina-a e redistribui a carga.

### 4.5 Análise de resultados
Reunimos todos os métodos numa só tabela para os comparar em **qualidade da solução** (km), **esforço de pesquisa** (nós expandidos, retrocessos ou iterações) e **tempo de execução**.

In [ ]:
def linha(nome, custo, esforco, tempo):
    return {"método": nome, "custo (km)": round(custo, 1), "esforço": esforco, "tempo (s)": round(tempo, 2)}

g_adm = resultados["Greedy | h admissível"]
g_inf = resultados["Greedy | h informativa"]
a_15  = resultados["A* ponderado (w=1.5) | h informativa"]
a_20  = resultados["A* ponderado (w=2) | h informativa"]
csp_s = csp_res["+ MRV + forward checking + LCV"]
nome15 = "A* ponderado (w=1.5) | h informativa"

tab = pd.DataFrame([
    linha("Greedy (h admissível)",            g_adm.cost, f"{g_adm.expanded} nós", g_adm.seconds),
    linha("Greedy (h informativa)",           g_inf.cost, f"{g_inf.expanded} nós", g_inf.seconds),
    linha("A* ponderado w=1,5 (h informativa)", a_15.cost, f"{a_15.expanded} nós", a_15.seconds),
    linha("A* ponderado w=2 (h informativa)", a_20.cost, f"{a_20.expanded} nós", a_20.seconds),
    linha("CSP (MRV + FC + LCV)",             csp_s.cost, f"{csp_s.backtracks} retrocessos", csp_s.seconds),
    linha("CSP + Hill Climbing",              csp_hc_hist[-1], f"{len(csp_hc_hist) - 1} iterações", csp_s.seconds + csp_hc_tempo),
    linha("Greedy (h admissível) + Hill Climbing",  hc_hist["Greedy | h admissível"][-1],  f"{len(hc_hist['Greedy | h admissível']) - 1} iterações", g_adm.seconds + hc_tempo["Greedy | h admissível"]),
    linha("Greedy (h informativa) + Hill Climbing", hc_hist["Greedy | h informativa"][-1], f"{len(hc_hist['Greedy | h informativa']) - 1} iterações", g_inf.seconds + hc_tempo["Greedy | h informativa"]),
    linha("A* ponderado w=1,5 + Hill Climbing",     hc_hist[nome15][-1],                   f"{len(hc_hist[nome15]) - 1} iterações", a_15.seconds + hc_tempo[nome15]),
]).sort_values("custo (km)").reset_index(drop=True)
display(tab)

plt.figure(figsize=(8.5, 4.2))
plt.barh(tab["método"][::-1], tab["custo (km)"][::-1], color="tab:blue")
plt.xlabel("custo total (km)")
plt.title("Comparação do custo por método (menor é melhor)")
plt.grid(axis="x", alpha=.3)
plt.tight_layout()
plt.show()

#### Análise crítica

**Qualidade das soluções.** O melhor resultado é o do **A\* ponderado (w = 1,5) seguido de Hill Climbing (1594,8 km)**, seguido do Greedy admissível com Hill Climbing (1600,9 km) e do CSP com Hill Climbing (1606,1 km). A diferença entre o melhor (1595 km) e o pior método (1983 km) é de cerca de 24%, o que mostra que **a escolha do método importa muito**. Os métodos que só constroem uma solução ficam todos acima dos 1660 km, e **juntar a pesquisa local reduz o custo a cerca de 1600 km, quase independentemente do ponto de partida**.

**Esforço e tempo.** Todos os métodos terminam em **menos de 2 segundos** no dataset *tiny*, exceto o A\* clássico, que gasta cerca de 1,5 a 2,5 segundos até esgotar o limite sem encontrar solução. O Greedy com heurística informativa é o mais económico (40 nós); o A\* ponderado também é muito eficiente (56 nós) e dá a melhor solução construtiva sem precisar de pesquisa local.

**O papel da heurística.** A heurística admissível é demasiado otimista (só vê o pior pedido isolado), por isso não consegue orientar a pesquisa e o A\* clássico não termina; nem aumentando o peso `w` ela chega a resolver. A informativa é mais realista e transforma o problema em algo tratável, ao custo de perder a garantia de otimalidade.

**O efeito da regra do armazém.** A obrigação de passar pelo armazém alonga todas as viagens (de cerca de 56 para cerca de 100 minutos por pedido), aperta as janelas de entrega e **torna 1 pedido impossível**. Cada veículo faz muitas idas ao armazém (3 e 9 na melhor solução), o que explica que o custo total seja alto para apenas 19 pedidos leves.

**Pontos fortes e fracos de cada abordagem.**
| Abordagem | Pontos fortes | Pontos fracos |
|---|---|---|
| **Greedy** | muito rápido, sempre encontra solução | qualidade fraca; sem garantias |
| **A\*** | com `w` bem escolhido dá boa solução com poucos nós | o A\* clássico não escala neste problema |
| **Hill Climbing** | grande melhoria com pouco código e tempo | fica preso em ótimos locais; depende da solução inicial |
| **CSP** | garante soluções viáveis, e as heurísticas reduzem retrocessos quando há conflitos | não otimiza a sequência; atribuição desequilibrada; aqui não há conflitos para mostrar o seu valor |

#### Limitações do trabalho
1. Não conhecemos a solução ótima, por isso não sabemos quão longe dela estamos.
2. Assumimos que o **mesmo veículo** recolhe e entrega; não modelámos transferências entre veículos nem tempo de armazenamento ou de carga/descarga.
3. O CSP não tem restrições de equilíbrio entre veículos, e neste dataset não gera conflitos.
4. O custo considera apenas distância, ignorando os tempos de espera (que chegam a ser de vários dias).
5. Os testes foram feitos só no dataset *tiny* e um pedido foi retirado por ser impossível.

#### Melhorias futuras
- **Heurística admissível mais forte**, que tenha em conta as janelas temporais (por exemplo, um limite inferior baseado numa árvore geradora mínima), para que o A\* clássico consiga terminar.
- **Vizinhanças mais ricas** na pesquisa local (trocar pedidos entre veículos, inverter troços de rota) e **Simulated Annealing**, para escapar de ótimos locais.
- **Melhor sequenciação no CSP**: otimizar a ordem das paragens de cada veículo e acrescentar uma restrição suave de equilíbrio de carga.
- **Armazém como "hub"**: permitir que um veículo deixe os produtos no armazém e outro os entregue mais tarde.
- **Custo mais completo**, que penalize os tempos de espera e use a velocidade de cada veículo.
- **Testar no dataset maior** e medir a escalabilidade de cada método.

### 4.6 A melhor solução, em detalhe
Escolhemos automaticamente a solução de menor custo entre as que terminam em pesquisa local e a **auditamos** de forma independente.

In [ ]:
candidatas = {f"{nome} + Hill Climbing": rotas for nome, rotas in hc_rotas.items()}
candidatas["CSP + Hill Climbing"] = csp_hc_rotas
melhor_nome = min(candidatas, key=lambda n: p.solution_cost(candidatas[n]))
melhor = candidatas[melhor_nome]

print(f"Melhor solução: {melhor_nome}")
print(f"Custo total: {p.solution_cost(melhor):.1f} km | solução completa: {p.is_complete(melhor)}")

tabela_auditoria, erros = auditar(p, melhor)
display(tabela_auditoria)
print("Auditoria independente:", "✔ sem violações" if not erros else erros)
desenhar_rotas(p, melhor, f"Melhor solução: {melhor_nome}")

**O que vemos.**
- A auditoria independente **não encontra nenhuma violação**: todas as janelas são cumpridas, a carga nunca excede a capacidade (o pico é de cerca de 37 kg contra mais de 1000 kg de capacidade), todos os pedidos são entregues e o custo calculado à parte coincide com o do verificador.
- A solução reparte os pedidos pelos dois veículos (7 e 12 pedidos, com cerca de 517 km e 1078 km) e os veículos fazem **3 e 9 idas forçadas ao armazém** por causa da regra. O agrupamento permite que algumas dessas idas sirvam vários pedidos ao mesmo tempo.
- As esperas são enormes (cerca de 14 e 11 dias em cada veículo, no total), porque os pedidos estão espalhados por duas semanas. No nosso modelo esperar é grátis, mas num cenário real isto obrigaria a reconsiderar o plano (por exemplo, usar o veículo noutras tarefas).

#### Horário completo da melhor solução

In [ ]:
horario = mostrar_rotas(p, melhor)
pd.set_option("display.max_rows", None)
horario

**Como ler a tabela.** Cada linha é uma paragem: `P` é uma recolha, `D` uma entrega e `armazém` uma passagem obrigatória pelo armazém. "Chegada" é a hora a que o veículo chega, "serviço" é a hora a que a operação ocorre (depois de esperar, se chegou antes de a janela abrir), "espera" são os minutos de espera e "carga" é o peso a bordo depois da operação.

---
## 5. Conclusão

Neste projeto desenvolvemos um agente inteligente que planeia rotas de recolha e entrega de produtos locais, em que os produtos têm de **passar pelo armazém antes de serem entregues**, respeitando janelas temporais e a capacidade dos veículos. Começámos por **explorar e validar os dados** (EDA), o que revelou que o dataset é limpo e consistente e que as **janelas temporais**, e não a capacidade, são o que realmente condiciona o problema. Seguiu-se a **modelação**: um estado, ações e custo bem definidos, um verificador de viabilidade que aplica todas as regras num único sítio, e duas heurísticas com propriedades diferentes. Por fim, **implementámos e comparámos** quatro técnicas: Greedy, A\*, Hill Climbing e CSP.

Os resultados mostram que:
- todos os métodos encontram **soluções completas e viáveis** para os 19 pedidos planeáveis, exceto o A\* clássico, que não termina; a regra do armazém torna ainda **1 pedido impossível**, que é retirado e reportado;
- a **heurística** tem enorme impacto: uma heurística mais informada reduz o esforço de pesquisa de forma drástica (de 892 para 40 nós no Greedy) e o A\* só se torna prático com a versão ponderada, sendo `w = 1,5` o melhor equilíbrio que encontrámos;
- a **pesquisa local** é a técnica que mais reduz o custo: a melhor solução obtida, **1594,8 km**, resultou de melhorar com Hill Climbing a solução do A\* ponderado, e partindo de qualquer das soluções construtivas chega-se a cerca de 1600 km;
- no **CSP**, a solução é encontrada sem retrocessos em todas as configurações, porque o dataset não gera conflitos; o valor das heurísticas MRV, forward checking e LCV só ficaria visível num dataset mais apertado;
- a combinação de métodos funciona melhor do que cada um isolado: **construir** uma solução (A\*, Greedy ou CSP) e **refiná-la** com pesquisa local.

Como lição principal, não há um método melhor em tudo: o Greedy é rápido, o A\* ponderado equilibra rapidez e qualidade, o CSP garante viabilidade e o Hill Climbing refina. A regra do armazém mostrou também que **uma decisão de modelação pode pesar tanto como a escolha do algoritmo**: alongou as viagens em cerca de 80% (de 56 para 100 minutos em média) e eliminou um pedido. Como trabalho futuro ficam uma heurística admissível mais forte, vizinhanças mais ricas na pesquisa local, um modelo de armazém como *hub* com transferências entre veículos e a avaliação no dataset maior.

---
## 6. Anexos

### 6.1 Estrutura da entrega
```
IA_P01_G##.zip
├── readme.txt           link do repositório GitHub
├── 01_eda.ipynb         análise exploratória
├── 02_relatorio.ipynb   este relatório
├── algoritmos.py        modelo e algoritmos
└── data_tiny/           dados (e data_large/ se aplicável)
```

### 6.2 Referências
- Slides da unidade curricular de Inteligência Artificial (enunciado do Projeto 01).
- S. Russell e P. Norvig, *Artificial Intelligence: A Modern Approach*: pesquisa informada (Greedy e A\*), pesquisa local (Hill Climbing) e problemas de satisfação de restrições.